**METRICS — Optimized joins on the 100M-row fact**

Every cell below applies the same playbook:

- **Project both sides before the join** — fact narrowed to ~7 columns; each dim narrowed to `(join_key + only the columns this metric uses)`.
- **`broadcast()` on every small dim** — forces a `BroadcastHashJoin`, no shuffle on the fact side.
- **Aggregate after enrichment** — partial aggregation runs inside each map task; only the small partial results shuffle to the final reducer.
- **`.explain("formatted")` printed for every plan** — verify `BroadcastHashJoin`, no `Exchange` on the fact side before the join.

**Shuffle reduction summary:** at 100M rows each avoided shuffle saves writing/reading ~5–10GB of intermediate data. Broadcasting the dims (each <1MB) trades a tiny per-executor memory cost for eliminating the fact-side shuffle entirely. Partial aggregation reduces the final shuffle to `group_cardinality × num_partitions` rows — typically a few thousand for region/category/date groupings.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

# Read RAW fact_sales (not pre-enriched). The point of this notebook is to demonstrate
# query-time join optimization — joining per-metric lets each metric project only the
# columns it actually needs from each dim. A pre-enriched table forces every metric to
# read every column, which wastes I/O at 100M-row scale.
fact_sales = spark.table("fact_sales")

# Read each dim once. Each metric below narrows them further before joining.
# Sizes (rows): dim_store=200, dim_product=5_000, dim_promotion=50.
# All three are tiny — well under spark.sql.autoBroadcastJoinThreshold (default 10MB).
# We never need dim_customer for any metric here, so we don't load it.
dim_store = spark.table("dim_store")
dim_product = spark.table("dim_product")
dim_promotion = spark.table("dim_promotion")

# Helper: net_revenue formula reused by every metric. Computed inline (no UDF) so Catalyst
# can fold it into the same Project node as the join output — zero extra stages.
#   net_revenue = quantity * unit_price * (1 - coalesce(discount_pct, 0))
# The coalesce handles the ~70% of rows where promotion_id is NULL → no promo row matches → discount_pct is NULL.
def net_revenue_expr():
    return F.col("quantity") * F.col("unit_price") * (F.lit(1.0) - F.coalesce(F.col("discount_pct"), F.lit(0.0)))

Revenue by region ->shows business performance by global region

In [0]:
# Revenue by region.
# Joins needed: fact -> dim_store (region), fact -> dim_promotion (discount_pct).
# We do NOT need dim_product for this metric, so it's not joined → no wasted shuffle/broadcast.

# Step 1: project the fact down to ONLY the columns we actually use.
# Without this, Spark would carry every fact column through the broadcast hash join,
# inflating each task's heap usage. At 100M rows, dropping unused columns saves ~30–40%
# of in-flight memory and lets the broadcast hash table stay tighter on each executor.
fact_proj = fact_sales.select(
    "store_id",        # join key for dim_store
    "promotion_id",    # join key for dim_promotion
    "quantity",        # net_revenue input
    "unit_price",      # net_revenue input
)

# Step 2: project each dim to (join_key + ONLY columns we need from it).
# dim_store has 4 columns; we only need region. The narrower the broadcast, the less
# memory each executor consumes for the broadcast hash table.
store_proj = dim_store.select("store_id", "region")
promo_proj = dim_promotion.select("promotion_id", "discount_pct")

# Step 3: broadcast joins. Both dims are tiny (<1MB each). broadcast() forces a
# BroadcastHashJoin, which means:
#   - dim is sent to every executor once (small one-time network cost),
#   - the 100M-row fact is NEVER shuffled — it streams through each task locally.
# Compare to a default sort-merge join, which would shuffle BOTH sides by the join key,
# i.e., ~5–10GB of fact data over the network per join. We're saving two such shuffles here.
revenue_by_region = (
    fact_proj
    .join(broadcast(store_proj), "store_id", "left")
    .join(broadcast(promo_proj), "promotion_id", "left")
    # Step 4: aggregate AFTER enrichment.
    # Spark performs partial aggregation (HashAggregate) inside each map task before the
    # final Exchange. So per-task we go from ~500K rows → ~5 rows (one per region).
    # The shuffle that feeds the final aggregation only carries num_partitions × 5 = 1000 rows.
    .groupBy("region")
    .agg(F.round(F.sum(net_revenue_expr()), 2).alias("total_net_revenue"))
    .orderBy(F.desc("total_net_revenue"))
)

# Inspect the physical plan. Look for:
#   - "BroadcastHashJoin" (not SortMergeJoin) on both join nodes
#   - "BroadcastExchange" feeding only the dim side of each join
#   - No "Exchange" on the fact side before the final HashAggregate's shuffle
#   - Two HashAggregate nodes: a partial one before the Exchange, a final one after
revenue_by_region.explain("formatted")
display(revenue_by_region)

revenue by product category

In [0]:
# Revenue by product category.
# Joins: fact -> dim_product (category), fact -> dim_promotion (discount_pct).
# dim_store is irrelevant here — skipping it removes a join entirely.

# Project fact to ONLY needed columns. Each dropped column means less data flowing through
# the join's output rows on the executor side.
fact_proj = fact_sales.select("product_id", "promotion_id", "quantity", "unit_price")

# Narrow dims: dim_product has 4 columns, we only need category. dim_promotion just needs
# discount_pct. Smaller broadcast hash tables = less per-executor memory.
product_proj = dim_product.select("product_id", "category")
promo_proj = dim_promotion.select("promotion_id", "discount_pct")

revenue_by_category = (
    fact_proj
    # broadcast() forces BroadcastHashJoin, eliminating the fact-side shuffle that a
    # SortMergeJoin would require (~5–10GB of network traffic at 100M rows).
    .join(broadcast(product_proj), "product_id", "left")
    .join(broadcast(promo_proj), "promotion_id", "left")
    # Aggregate after enrichment. Partial aggregation collapses 100M rows down to
    # 8 (one per category) PER TASK, so the shuffle to the final aggregator carries
    # num_partitions × 8 ≈ 1.6K rows total — negligible compared to the unaggregated case.
    .groupBy("category")
    .agg(F.round(F.sum(net_revenue_expr()), 2).alias("total_net_revenue"))
    .orderBy(F.desc("total_net_revenue"))
)

# Verify in the plan: BroadcastHashJoin on both joins, BroadcastExchange only on dim sides,
# no Exchange on fact side before the partial HashAggregate.
revenue_by_category.explain("formatted")
display(revenue_by_category)

Gross margin by category

In [0]:
# Gross margin by category.
# Joins: fact -> dim_product (category, unit_cost), fact -> dim_promotion (discount_pct).
# Both gross_margin and net_revenue need promo for the discount, and product for unit_cost.

fact_proj = fact_sales.select("product_id", "promotion_id", "quantity", "unit_price")

# dim_product needs TWO columns this time: category for the group, unit_cost for the math.
# Still tiny (~5K rows × 2 cols ≈ a few hundred KB) — well under broadcast threshold.
product_proj = dim_product.select("product_id", "category", "unit_cost")
promo_proj = dim_promotion.select("promotion_id", "discount_pct")

margin_by_category = (
    fact_proj
    .join(broadcast(product_proj), "product_id", "left")
    .join(broadcast(promo_proj), "promotion_id", "left")
    # Compute net_revenue and cost in a single Project (Catalyst fuses these withColumns).
    # Doing both metric inputs in one pass means we don't re-scan the row to compute them
    # separately — they share the per-row read of quantity, unit_price, unit_cost.
    .withColumn("net_revenue", net_revenue_expr())
    .withColumn("cost", F.col("quantity") * F.col("unit_cost"))
    # Aggregate after enrichment. Partial HashAggregate collapses each task to 8 rows
    # (one per category), so the shuffle is trivial. We compute three sums in the same
    # aggregation pass so we only scan the post-join rows once.
    .groupBy("category")
    .agg(
        F.round(F.sum("net_revenue"), 2).alias("total_net_revenue"),
        F.round(F.sum(F.col("net_revenue") - F.col("cost")), 2).alias("gross_margin"),
        # gross_margin_pct computed from the two sums above. Doing the division INSIDE
        # the agg(...) lets Catalyst share the partial sums of net_revenue and (net_revenue-cost)
        # — no need for a self-join or window function.
        F.round(F.sum(F.col("net_revenue") - F.col("cost")) / F.sum("net_revenue"), 4).alias("gross_margin_pct"),
    )
    .orderBy(F.desc("gross_margin"))
)

margin_by_category.explain("formatted")
display(margin_by_category)

Average basket value

In [0]:
# Average basket value.
# In our schema each transaction_id is unique → "basket_revenue per transaction" is just
# the row's net_revenue. The original cell did a groupBy("transaction_id") with sum() — at
# 100M unique keys that's a 100M-group shuffle, which is enormous and pointless.
#
# Optimization: skip the per-transaction groupBy entirely and compute the global avg
# in a single full-table aggregation. That's a single shuffle of partial sums (~200 rows
# total), regardless of fact size.

# Project to bare minimum: we only need promotion_id (for discount lookup) plus the two
# numeric inputs. No store/product/customer joins needed — none of those affect basket value.
fact_proj = fact_sales.select("promotion_id", "quantity", "unit_price")

# dim_promotion is the only join — broadcast it to avoid shuffling the 100M-row fact.
promo_proj = dim_promotion.select("promotion_id", "discount_pct")

avg_basket_value = (
    fact_proj
    .join(broadcast(promo_proj), "promotion_id", "left")
    # Single global aggregation. agg() with no groupBy() = one row out, but partial
    # aggregation still runs per task, so the shuffle to the final reducer is just
    # num_partitions × 1 = ~200 rows total.
    .agg(F.round(F.avg(net_revenue_expr()), 2).alias("avg_basket_value"))
)

# Plan should show: BroadcastHashJoin → partial HashAggregate (per task) → Exchange
# (tiny, ~200 rows of partial sums) → final HashAggregate. No fact-side shuffle anywhere.
avg_basket_value.explain("formatted")
display(avg_basket_value)

Daily revenue trend

In [0]:
# Daily revenue trend.
# Joins: fact -> dim_promotion (discount_pct).
# transaction_date already lives on the fact, so no date-dim join needed. We don't
# touch dim_store/dim_product/dim_customer — none of them affect a daily revenue total.

# Project fact to ONLY needed columns. transaction_date is the partition column on the
# Delta write, so reading just it + the three measure-input columns lets Spark prune
# everything else in the columnar Parquet read — significant I/O savings at 100M rows.
fact_proj = fact_sales.select("transaction_date", "promotion_id", "quantity", "unit_price")

# Narrow dim_promotion to (join_key, discount_pct). 50 rows × 2 cols = trivially broadcastable.
promo_proj = dim_promotion.select("promotion_id", "discount_pct")

daily_revenue = (
    fact_proj
    # broadcast() forces BroadcastHashJoin → no shuffle of the 100M-row fact.
    # A default sort-merge join here would shuffle ~5GB of fact data; we save that entirely.
    .join(broadcast(promo_proj), "promotion_id", "left")
    # Aggregate after enrichment. Partial HashAggregate collapses each task to ~365 rows
    # (one per date in CY2024), so the shuffle to the final aggregator carries
    # num_partitions × 365 ≈ 73K rows — tiny compared to 100M unaggregated rows.
    .groupBy("transaction_date")
    .agg(F.round(F.sum(net_revenue_expr()), 2).alias("daily_net_revenue"))
    .orderBy("transaction_date")
)

# Verify in plan: BroadcastHashJoin (not SortMergeJoin), BroadcastExchange only on dim side,
# no Exchange on fact side before the partial HashAggregate, then the final tiny shuffle.
daily_revenue.explain("formatted")
display(daily_revenue)


Top 10 stores by revenue

In [0]:
# Top 10 stores by revenue.
# Joins: fact -> dim_store (country, region, store_type), fact -> dim_promotion (discount_pct).
# We don't touch dim_product/dim_customer — neither affects a per-store revenue rollup.
#
# Optimization note: we group by store_id FIRST (revenue is determined entirely by store_id),
# THEN join the store attributes onto the small post-aggregation result. This avoids carrying
# country/region/store_type through the 100M-row join — they only need to ride along with
# the 200 aggregated store rows at the end.

fact_proj = fact_sales.select("store_id", "promotion_id", "quantity", "unit_price")

# dim_promotion narrowed to (join_key, discount_pct). dim_store gets joined LATER (after agg)
# so we don't even need to broadcast it through the 100M-row stage.
promo_proj = dim_promotion.select("promotion_id", "discount_pct")
store_proj = dim_store.select("store_id", "country", "region", "store_type")

# Step 1: enrich with discount, aggregate to per-store totals (200 rows out).
# This is the only stage that touches the 100M-row fact — and it only carries 5 columns.
per_store_revenue = (
    fact_proj
    .join(broadcast(promo_proj), "promotion_id", "left")
    # Partial HashAggregate collapses each task to ~200 rows (one per store_id).
    # Final shuffle: num_partitions × 200 ≈ 40K rows, vs 100M unaggregated.
    .groupBy("store_id")
    .agg(F.round(F.sum(net_revenue_expr()), 2).alias("total_net_revenue"))
)

# Step 2: join store attributes onto the 200-row result. At this scale the broadcast is
# essentially free — and crucially, country/region/store_type never traveled through the
# big-fact join. This is a pure metadata enrichment on a tiny intermediate result.
top_10_stores = (
    per_store_revenue
    .join(broadcast(store_proj), "store_id", "left")
    .orderBy(F.desc("total_net_revenue"))
    .limit(10)
)

# Plan should show: BroadcastHashJoin with promotion → partial HashAggregate (per task,
# ~200 rows) → small Exchange → final HashAggregate → BroadcastHashJoin with store
# (200×200 rows, trivial) → TakeOrderedAndProject for the top-10. No fact-side shuffle.
top_10_stores.explain("formatted")
display(top_10_stores)


Promotion performance

In [0]:
# Promotion performance.
# Joins: fact -> dim_promotion (promotion_type, discount_pct).
# Single join — store/product/customer don't affect a promotion-type rollup.
#
# Subtlety: 70% of fact rows have NULL promotion_id (most transactions aren't promoted).
# A LEFT join keeps those rows; their promotion_type/discount_pct will be NULL and we
# group them under a "no_promotion" bucket so they show up in the report rather than
# being silently dropped (which an inner join would do).

# Project to ONLY needed columns. Carrying anything else through the join would inflate
# the post-join row width unnecessarily.
fact_proj = fact_sales.select("promotion_id", "quantity", "unit_price")

# dim_promotion narrowed to the two columns this metric uses. 50 rows × 3 cols, trivial broadcast.
promo_proj = dim_promotion.select("promotion_id", "promotion_type", "discount_pct")

promotion_performance = (
    fact_proj
    # broadcast() forces BroadcastHashJoin → no shuffle of the 100M-row fact.
    # SortMergeJoin alternative would shuffle ~5GB of fact data on promotion_id.
    .join(broadcast(promo_proj), "promotion_id", "left")
    # Compute gross_revenue / discount_amount / net_revenue once in a single Project
    # so all three sums share the same per-row scan. coalesce() handles unpromoted rows
    # (NULL discount_pct → 0). Catalyst fuses these withColumn calls into one project node.
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
    .withColumn(
        "discount_amount",
        F.col("gross_revenue") * F.coalesce(F.col("discount_pct"), F.lit(0.0)),
    )
    .withColumn("net_revenue", F.col("gross_revenue") - F.col("discount_amount"))
    # Bucket NULL promotion_type as "no_promotion" so unpromoted rows show up explicitly
    # rather than getting a NULL group key (which display() will render as a blank row).
    .withColumn("promotion_type", F.coalesce(F.col("promotion_type"), F.lit("no_promotion")))
    # Aggregate after enrichment. promotion_type has 6 distinct values (5 real + no_promotion),
    # so partial HashAggregate collapses each task to 6 rows. Final shuffle:
    # num_partitions × 6 ≈ 1.2K rows — negligible.
    # All four aggregations share the same scan of the post-join rows.
    .groupBy("promotion_type")
    .agg(
        F.count("*").alias("transaction_lines"),
        F.round(F.sum("gross_revenue"), 2).alias("gross_revenue"),
        F.round(F.sum("discount_amount"), 2).alias("total_discount"),
        F.round(F.sum("net_revenue"), 2).alias("net_revenue"),
    )
    .orderBy(F.desc("net_revenue"))
)

# Plan should show: BroadcastHashJoin → partial HashAggregate → small Exchange → final
# HashAggregate. No SortMergeJoin, no fact-side shuffle before the partial aggregation.
promotion_performance.explain("formatted")
display(promotion_performance)


## Skew handling — `store_id` is hot

The fact generator deliberately sends ~50% of all transactions to `store_id = 1`. That's a textbook **hot key**: a single value that dominates volume and breaks the assumption Spark relies on for balanced shuffles.

**Why it matters at 100M rows:**

A `groupBy("store_id").sum(...)` runs in two physical stages:
1. **Partial aggregation** — each task aggregates locally. This is fine; every task just produces ~200 rows (one per store) regardless of skew.
2. **Final aggregation** — partial results are shuffled by `store_id` so all rows with the same key land on the same reducer. **This is where skew bites.** With ~50M rows worth of partials all hashing to the same `store_id=1`, one reducer task receives wildly more data than its peers. The job's wall-clock time is bounded by that one straggler.

For *count*-style sums where the partial aggregation already collapses each task to ~200 rows, the skew impact is small — the final shuffle only carries `num_partitions × num_keys` rows total. The pathological case is when skew survives a wider operation (joins on the skewed key, distinct counts, window functions partitioned by the skewed key), where one reducer must process the full ~50M-row slice.

**Detection then mitigation:** below we (1) measure the skew quantitatively, then (2) demonstrate the *salting* technique — a general fix that splits the hot key into N synthetic sub-keys, runs the heavy aggregation per-salt, then sums the partial results.

In [ ]:
# Skew detection: measure the row-count distribution across store_id.
# Two complementary signals:
#   1. The "hot key share" — what fraction of all rows belong to the single most common key.
#      A uniform distribution across 200 stores would give ~0.5%; anything an order of magnitude
#      higher is a red flag.
#   2. Top-vs-median ratio — how much heavier is the hottest key vs. a typical key.
#      Uniform → ~1x. Pathological skew → 100x+.

# Step 1: per-key row counts. This is itself a groupBy on the skewed key, but count(*) only
# carries 8 bytes per partial result, so the final shuffle is num_partitions × num_keys rows
# (≈ 40K) — small enough that skew on the key doesn't meaningfully hurt this diagnostic query.
store_counts = (
    fact_sales
    .groupBy("store_id")
    .agg(F.count("*").alias("row_count"))
)

# Step 2: compute summary statistics in a single pass. Doing this as one aggregation (no
# groupBy) means a single global shuffle of partial results — irrelevant cost.
total_rows = fact_sales.count()  # cheap on Delta — uses file-level row-count metadata

skew_summary = (
    store_counts
    .agg(
        F.max("row_count").alias("max_rows"),
        F.expr("percentile_approx(row_count, 0.5)").alias("median_rows"),
        F.min("row_count").alias("min_rows"),
        F.count("*").alias("num_distinct_stores"),
    )
    .withColumn("hot_key_share_pct", F.round(F.col("max_rows") / F.lit(total_rows) * F.lit(100), 2))
    .withColumn("max_to_median_ratio", F.round(F.col("max_rows") / F.col("median_rows"), 1))
)

print(f"Total fact rows: {total_rows:,}")
print("Skew summary:")
display(skew_summary)

# Step 3: top-5 hottest keys, so the offender is named, not just summarized.
# A uniform 200-store distribution would have 100M / 200 = 500K rows/store. Anything visibly
# above that is hotter than expected; store_id=1 should land at ~50M (100x the uniform expectation).
print("Top 5 stores by row count (hot keys):")
display(store_counts.orderBy(F.desc("row_count")).limit(5))


### Salted aggregation for revenue-by-store

**The technique in three lines:**
1. Append a random salt `0..N-1` to the skewed key, turning `store_id=1` into N synthetic sub-keys: `(1, 0), (1, 1), …, (1, N-1)`. Spark's hash partitioner now spreads those across N reducers instead of piling them onto one.
2. Aggregate by the **composite** `(store_id, salt)` key. Each reducer handles `~hot_rows / N` rows from the hot key — load is balanced.
3. Drop the salt and aggregate again by `store_id` alone. The second aggregation runs over a tiny intermediate (`200 × N` rows), so it's effectively free.

**When salting is worth it:** when *one* reducer is the straggler. If your cluster has 200 cores and one reducer takes 10x as long as the others, salting with `N=10` flattens the histogram so all 10 sub-tasks finish in roughly the time of the previous fast reducers — wall-clock drops by ~10x for the heavy stage.

**Cost:** salting forces an extra aggregation stage. For workloads where the hot key isn't actually the bottleneck (e.g. our `revenue_by_region` cell — partial aggregation already collapses each task to 5 rows before the shuffle), salting adds work without removing a real bottleneck. **Always profile before salting.**

**Choosing N:** rule of thumb is `N ≈ (hot_key_share × num_partitions) / target_share_per_task`. With 50% hot share on 200 partitions and a target of ~1% per task, `N ≈ 50`. We use `N=16` below — a moderate value that demonstrates the technique without over-fragmenting the small keys.

In [ ]:
# Salted revenue-by-store aggregation.
# Pattern: two-stage aggregation — first by (store_id, salt), then by store_id alone.

NUM_SALTS = 16  # see markdown above for sizing rationale.

# Project to the columns this metric uses. Same playbook as the other metric cells.
fact_proj = fact_sales.select("store_id", "promotion_id", "quantity", "unit_price")
promo_proj = dim_promotion.select("promotion_id", "discount_pct")

# Step 1: enrich with discount via broadcast join (no fact-side shuffle).
enriched = (
    fact_proj
    .join(broadcast(promo_proj), "promotion_id", "left")
    # Compute net_revenue inline so the salted aggregation has a single numeric column to sum.
    .withColumn("net_revenue", net_revenue_expr())
    # Add a deterministic-but-uniform salt in [0, NUM_SALTS). pmod(rand()*N) gives an
    # integer in [0, N). We use rand() (not hash(transaction_id)) because the salt only
    # needs to be uniform — it doesn't need to be reproducible across runs, and rand() is
    # cheaper than a hash. Seed makes the assignment deterministic for testing.
    .withColumn("salt", F.pmod((F.rand(seed=99) * F.lit(NUM_SALTS)).cast("int"), F.lit(NUM_SALTS)))
)

# Step 2: FIRST aggregation — by the composite (store_id, salt) key.
# This is the heavy stage: it processes 100M rows. With salting, the hot store_id=1 is now
# spread across NUM_SALTS sub-keys (1,0), (1,1), ..., (1,15). Each receives ~50M/16 ≈ 3.1M
# rows of partial sums, which the hash partitioner distributes across distinct reducers.
# Without salting, all ~50M rows for store_id=1 funnel to a single reducer.
#
# Output cardinality: NUM_STORES × NUM_SALTS = 200 × 16 = 3,200 rows. Tiny.
salted_partial = (
    enriched
    .groupBy("store_id", "salt")
    .agg(F.sum("net_revenue").alias("salted_revenue"))
)

# Step 3: SECOND aggregation — collapse the salt dimension.
# Input is the 3,200-row intermediate from step 2. The shuffle here is ~num_partitions × 200
# = 40K rows of partial sums. Since the input is already aggregated, this stage finishes
# instantly regardless of the original skew.
revenue_by_store_salted = (
    salted_partial
    .groupBy("store_id")
    .agg(F.round(F.sum("salted_revenue"), 2).alias("total_net_revenue"))
    .orderBy(F.desc("total_net_revenue"))
)

# Inspect the plan. The key things to verify:
#   - BroadcastHashJoin on promotion (no fact-side shuffle).
#   - TWO HashAggregate pairs (partial + final), one for the salted groupBy, one for the
#     unsalted re-aggregation. The intermediate Exchange between them carries ~3,200 rows
#     across NUM_SALTS distinct partition slots — load-balanced even though the underlying
#     fact is heavily skewed.
#   - Compare against the un-salted top_10_stores plan above: same join structure, but its
#     final-aggregation Exchange would funnel ~50M rows of partials onto one reducer for
#     store_id=1. The salted version replaces that one fat task with NUM_SALTS even ones.
revenue_by_store_salted.explain("formatted")
display(revenue_by_store_salted.limit(10))
